In [3]:
import pandas as pd
import numpy as np

np.random.seed(42)

months = pd.date_range(
    start="2025-01-01",
    periods=24,
    freq="MS"
)

dealers = {
    "Northeast": ["NE_D01", "NE_D02"],
    "South": ["SO_D01", "SO_D02"],
    "Midwest": ["MW_D01", "MW_D02"],
    "West": ["WE_D01", "WE_D02"],
}

vehicle_models = [
    "SUV",
    "Sedan",
    "EV",
    "Pickup",
]

base_demand = {
    "SUV": 120,
    "Sedan": 75,
    "EV": 85,
    "Pickup": 100,
}

region_effect = {
    "Northeast": {
        "SUV": 1.00,
        "Sedan": 1.05,
        "EV": 1.10,
        "Pickup": 0.80,
    },
    "South": {
        "SUV": 1.05,
        "Sedan": 0.90,
        "EV": 0.85,
        "Pickup": 1.20,
    },
    "Midwest": {
        "SUV": 1.00,
        "Sedan": 0.90,
        "EV": 0.80,
        "Pickup": 1.25,
    },
    "West": {
        "SUV": 1.05,
        "Sedan": 0.95,
        "EV": 1.25,
        "Pickup": 0.90,
    },
}

seasonality = {
    1: 0.90,
    2: 0.92,
    3: 1.00,
    4: 1.03,
    5: 1.05,
    6: 1.08,
    7: 1.05,
    8: 1.03,
    9: 1.00,
    10: 0.98,
    11: 1.02,
    12: 1.10,
}

rows = []

for month in months:
    month_number = month.month

    for region, dealer_list in dealers.items():
        for dealer_id in dealer_list:
            for vehicle_model in vehicle_models:

                expected_demand = (
                    base_demand[vehicle_model]
                    * region_effect[region][vehicle_model]
                    * seasonality[month_number]
                )

                forecast_variation = np.random.normal(
                    loc=1.0,
                    scale=0.06
                )

                forecast_units = round(
                    expected_demand * forecast_variation
                )

                forecast_units = max(forecast_units, 0)

                sales_variation = np.random.normal(
                    loc=1.0,
                    scale=0.08
                )

                sales_units = round(
                    expected_demand * sales_variation
                )

                sales_units = max(sales_units, 0)

                rows.append({
                    "month": month,
                    "region": region,
                    "dealer_id": dealer_id,
                    "vehicle_model": vehicle_model,
                    "forecast_units": forecast_units,
                    "sales_units": sales_units,
                })

synthetic = pd.DataFrame(rows)

synthetic["forecast_error_units"] = (
    synthetic["sales_units"]
    - synthetic["forecast_units"]
)

synthetic["forecast_error_pct"] = (
    synthetic["forecast_error_units"]
    / synthetic["forecast_units"]
    * 100
)

synthetic["absolute_forecast_error_pct"] = (
    synthetic["forecast_error_pct"].abs()
)

mape = synthetic["absolute_forecast_error_pct"].mean()

print(
    "\nMean Absolute Percentage Error:",
    round(mape, 2),
    "%"
)

print(synthetic.head(20))
print("\nShape:", synthetic.shape)

synthetic.to_csv(
    "data/synthetic/synthetic_operations.csv",
    index=False
)




Mean Absolute Percentage Error: 8.11 %
        month     region dealer_id vehicle_model  forecast_units  sales_units  \
0  2025-01-01  Northeast    NE_D01           SUV             111          107   
1  2025-01-01  Northeast    NE_D01         Sedan              74           80   
2  2025-01-01  Northeast    NE_D01            EV              83           83   
3  2025-01-01  Northeast    NE_D01        Pickup              79           76   
4  2025-01-01  Northeast    NE_D02           SUV             105          113   
5  2025-01-01  Northeast    NE_D02         Sedan              69           68   
6  2025-01-01  Northeast    NE_D02            EV              85           71   
7  2025-01-01  Northeast    NE_D02        Pickup              65           69   
8  2025-01-01      South    SO_D01           SUV             107          116   
9  2025-01-01      South    SO_D01         Sedan              57           54   
10 2025-01-01      South    SO_D01            EV              71     

OSError: Cannot save file into a non-existent directory: 'data/synthetic'

In [4]:
print(
    synthetic[
        [
            "month",
            "region",
            "dealer_id",
            "vehicle_model",
            "forecast_units",
            "sales_units",
            "forecast_error_units",
            "forecast_error_pct"
        ]
    ]
    .head(20)
    .round(2)
    .to_string(index=False)
)


     month    region dealer_id vehicle_model  forecast_units  sales_units  forecast_error_units  forecast_error_pct
2025-01-01 Northeast    NE_D01           SUV             111          107                    -4               -3.60
2025-01-01 Northeast    NE_D01         Sedan              74           80                     6                8.11
2025-01-01 Northeast    NE_D01            EV              83           83                     0                0.00
2025-01-01 Northeast    NE_D01        Pickup              79           76                    -3               -3.80
2025-01-01 Northeast    NE_D02           SUV             105          113                     8                7.62
2025-01-01 Northeast    NE_D02         Sedan              69           68                    -1               -1.45
2025-01-01 Northeast    NE_D02            EV              85           71                   -14              -16.47
2025-01-01 Northeast    NE_D02        Pickup              65           6

In [5]:
print(
    "\nMAPE by Vehicle Model:"
)

print(
    synthetic
    .groupby("vehicle_model")["absolute_forecast_error_pct"]
    .mean()
    .round(2)
)


MAPE by Vehicle Model:
vehicle_model
EV        8.47
Pickup    8.08
SUV       7.67
Sedan     8.24
Name: absolute_forecast_error_pct, dtype: float64


In [6]:
print(
    "\nMAPE by Region:"
)

print(
    synthetic
    .groupby("region")["absolute_forecast_error_pct"]
    .mean()
    .round(2)
)


MAPE by Region:
region
Midwest      7.63
Northeast    8.81
South        8.02
West         7.99
Name: absolute_forecast_error_pct, dtype: float64
